# 🇨🇭 Swiss Health Insurance – Premium Region Classification
### HEC Lausanne – Master Business Analytics | Machine Learning Project

**Research Question:** Can a Machine Learning model predict the official premium region (1, 2 or 3) of a Swiss commune from its socio-demographic profile — and detect communes that are misclassified by the politically frozen OFSP system?

---
### Data Sources
| File | Content | Role |
|---|---|---|
| `Anhang_EDI_Ver_2026.xlsx` | OFSP: commune → premium region (2026) | **Target Y** |
| `data.xlsx` | OFS socio-demographic stats per commune (2024) | **Features X** |
| `Prämien_CH.xlsx` | Premium prices by insurer/canton/region (2025) | **EDA enrichment** |
| `Prämien_CH__1_.xlsx` | Same for 2026 | **EDA enrichment** |

### Key design decision — join_key
In the premium files, the region code is `PR-REG CH0/1/2/3`:
- `CH0` = **single-region canton** (GE, AG, BS, etc.) → the only region, maps to `premium_region = 1`
- `CH1/2/3` = region 1/2/3 within a **multi-region canton** (BE, ZH, VD, etc.)

We cannot use `premium_region` alone as join key because GE (single, region 1) and BE zone 1 (multi, region 1) are **different market contexts**. We use `canton + region_code_suffix` as the correct join key.

## 0. Setup & Paths

In [17]:
# Install if needed
# !pip install openpyxl scikit-learn matplotlib seaborn --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Libraries loaded')

✅ Libraries loaded


In [21]:
# ─── LOCAL PATHS — adjust to your machine ───
# Place all xlsx files in a 'data/' folder next to this notebook
BASE       = r'C:\Users\mahde\OneDrive\Desktop\ML_Project\data' + os.sep
OUTPUT_DIR = r'C:\Users\mahde\OneDrive\Desktop\ML_Project\outputs' + os.sep
os.makedirs(BASE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_REGIONS_2026 = BASE + 'Anhang EDI Ver. über die PrReg_Mut_2026'
PATH_OFS_STATS    = BASE + 'data.xlsx'
PATH_PRIMES_2025  = BASE + 'Prämien_CH_2025.xlsx'
PATH_PRIMES_2026  = BASE + 'Prämien_CH_2026.xlsx'
OUTPUT_CSV        = OUTPUT_DIR + 'swiss_communes_ml_ready.csv'

# Cantons with a single premium region in the OFSP system
# These appear as PR-REG CH0 in the premium files
CH0_CANTONS = ['AG','AI','AR','BS','GE','GL','JU','NE','NW','OW','SO','SZ','TG','UR','ZG']

print(f'✅ Working directory : {os.getcwd()}')
print(f'   Data folder       : {BASE}')
print(f'   Outputs folder    : {OUTPUT_DIR}')

✅ Working directory : c:\Users\mahde\OneDrive\Desktop\ML_Project\notebooks
   Data folder       : C:\Users\mahde\OneDrive\Desktop\ML_Project\data\
   Outputs folder    : C:\Users\mahde\OneDrive\Desktop\ML_Project\outputs\


In [22]:
for f in [PATH_REGIONS_2026, PATH_OFS_STATS, PATH_PRIMES_2025, PATH_PRIMES_2026]:
    print(f'{"✅" if os.path.exists(f) else "❌ MANQUANT"} {os.path.basename(f)}')


❌ MANQUANT Anhang EDI Ver. über die PrReg_Mut_2026
✅ data.xlsx
✅ Prämien_CH_2025.xlsx
✅ Prämien_CH_2026.xlsx


## 1. Load OFSP Region Labels (Target Y)

Source: EDI Ordinance, valid 01.01.2026.  
Only **multi-region cantons** appear here (BE, ZH, VD, etc.).  
Single-region cantons (CH0) are assigned `premium_region = 1` in Step 3.

In [23]:
def load_region_labels(filepath):
    """Parse OFSP ordinance and build the join_key.
    
    join_key logic:
      - Single-region canton (CH0): join_key = 'CANTON_H0'
      - Multi-region canton:        join_key = 'CANTON_H{region_number}'
    
    This ensures GE (single, premium_region=1) maps to GE_H0
    and BE zone 1 (multi, premium_region=1) maps to BE_H1.
    They are different market contexts and must NOT be merged.
    """
    wb = load_workbook(filepath, read_only=True, data_only=True)
    ws = wb['Anhang EDI Ver. über die PR']
    rows = []
    for row in ws.iter_rows(values_only=True):
        if (row[1] is not None and isinstance(row[1], (int, float))
                and row[0] and len(str(row[0]).strip()) <= 4):
            canton = str(row[0]).strip()
            bfs    = int(row[1])
            name   = str(row[2]).strip() if row[2] else ''
            region = int(row[3]) if row[3] else None
            if canton and region:
                join_key = f'{canton}_H0' if canton in CH0_CANTONS else f'{canton}_H{region}'
                rows.append({
                    'canton': canton, 'bfs_nr': bfs,
                    'commune': name, 'premium_region': region,
                    'join_key': join_key
                })
    df = pd.DataFrame(rows)
    print(f'✅ Labels: {len(df)} communes | {df["premium_region"].value_counts().sort_index().to_dict()}')
    print(f'   join_keys (sample): {sorted(df["join_key"].unique())[:8]}')
    return df


def load_mutations(filepath):
    """Build fusion map: old BFS → new BFS."""
    wb  = load_workbook(filepath, read_only=True, data_only=True)
    ws  = wb['Mutationen']
    fusion_map = {}
    for row in ws.iter_rows(values_only=True):
        if not isinstance(row[0], (int, float)): continue
        mutation = str(row[3]).strip() if row[3] else ''
        if mutation.lower().startswith('fus'):
            parts = mutation.split()
            if len(parts) >= 2 and parts[1].isdigit():
                fusion_map[int(row[0])] = int(parts[1])
    print(f'✅ Mutations: {len(fusion_map)} commune fusions')
    return fusion_map


df_labels  = load_region_labels(PATH_REGIONS_2026)
fusion_map = load_mutations(PATH_REGIONS_2026)

# BFS → canton lookup (needed for single-region cantons)
bfs_canton_lookup = dict(zip(df_labels['bfs_nr'], df_labels['canton']))

InvalidFileException: openpyxl does not support . über die prreg_mut_2026 file format, please check you can open it with Excel first. Supported formats are: .xlsx,.xlsm,.xltx,.xltm

## 2. Load OFS Socio-Demographic Features (X)

In [ ]:
def load_ofs_stats(filepath, fusion_map):
    """Load OFS commune stats, rename columns, apply fusion map."""
    df = pd.read_excel(filepath, sheet_name='Data', header=3, engine='openpyxl')
    df = df.dropna(how='all')

    # Rename by position (safer than matching long French strings)
    col_map = {
        df.columns[0]:  'bfs_nr',
        df.columns[1]:  'commune_ofs',
        df.columns[2]:  'pop_density_2024',
        df.columns[3]:  'pct_hh_1p',
        df.columns[4]:  'pct_hh_2p',
        df.columns[5]:  'pct_hh_3p',
        df.columns[6]:  'pct_hh_4p',
        df.columns[7]:  'pct_hh_5p_plus',
        df.columns[8]:  'social_assistance_rate',
        df.columns[9]:  'sa_estimated',
        df.columns[10]: 'social_assistance_count',
        df.columns[11]: 'sa_count_est',
    }
    df = df.rename(columns=col_map)

    # BFS to int
    df['bfs_nr'] = pd.to_numeric(df['bfs_nr'], errors='coerce')
    df = df.dropna(subset=['bfs_nr'])
    df['bfs_nr'] = df['bfs_nr'].astype(int)

    # Apply fusion map (old BFS → new BFS)
    df['bfs_nr'] = df['bfs_nr'].map(lambda x: fusion_map.get(x, x))

    # Numeric conversion
    num_cols = ['pop_density_2024','pct_hh_1p','pct_hh_2p','pct_hh_3p',
                'pct_hh_4p','pct_hh_5p_plus','social_assistance_rate','social_assistance_count']
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Estimation flags
    for c in ['sa_estimated', 'sa_count_est']:
        df[c] = df[c].apply(lambda x: 1 if str(x).strip() == '*' else 0)

    # Aggregate fused communes
    agg_rules = {c: 'mean' for c in num_cols}
    agg_rules['social_assistance_count'] = 'sum'
    agg_rules['sa_estimated']  = 'max'
    agg_rules['sa_count_est']  = 'max'
    agg_rules['commune_ofs']   = 'first'
    before = len(df)
    df = df.groupby('bfs_nr').agg(agg_rules).reset_index()
    print(f'✅ OFS Stats: {len(df)} communes ({before - len(df)} fused)')
    return df


df_ofs = load_ofs_stats(PATH_OFS_STATS, fusion_map)
display(df_ofs.head(5))

## 3. Load & Aggregate Premium Market Features

**Standard adult profile:** `AKL-ERW` + `TAR-BASE` + `FRAST1` + `isBaseF=1`  
**join_key:** `canton + '_H' + last digit of region code`  
→ `GE_H0` (Genève, single region) vs `BE_H1` (Berne, zone 1) are kept **separate**

**Features computed (robust — no raw min/max):**
- `avg_premium` — mean across all insurers  
- `median_premium` — median (robust to outlier insurers)  
- `std_premium` — price dispersion  
- `p10_premium` — what a price-conscious person pays  
- `p90_premium` — what someone who doesn't compare pays  
- `iqr_premium` — interquartile range (competition proxy, no outliers)  
- `n_insurers` — number of active insurers in that market

In [ ]:
def build_premium_features(filepath, year_label):
    """Compute premium market features per canton×region.
    
    Excludes ZE and ZR (invalid canton codes with implausible prices).
    join_key = canton + '_H' + last digit of PR-REG code:
      PR-REG CH0 → _H0  (single-region canton)
      PR-REG CH1 → _H1  (zone 1 of multi-region canton)
    """
    df = pd.read_excel(filepath, sheet_name='Export')
    base = df[
        (df['Altersklasse']   == 'AKL-ERW') &
        (df['Tariftyp']       == 'TAR-BASE') &
        (df['Franchisestufe'] == 'FRAST1') &
        (df['isBaseF']        == 1) &
        (~df['Kanton'].isin(['ZE', 'ZR']))
    ].copy()

    # Correct join_key: canton + '_H' + last digit of region code
    base['join_key'] = base['Kanton'] + '_H' + base['Region'].str[-1]

    agg = base.groupby(['Kanton', 'join_key']).agg(
        avg_premium    = ('Prämie', 'mean'),
        median_premium = ('Prämie', 'median'),
        std_premium    = ('Prämie', 'std'),
        p10_premium    = ('Prämie', lambda x: x.quantile(0.10)),
        p90_premium    = ('Prämie', lambda x: x.quantile(0.90)),
        iqr_premium    = ('Prämie', lambda x: x.quantile(0.75) - x.quantile(0.25)),
        n_insurers     = ('Versicherer', 'nunique')
    ).reset_index().rename(columns={'Kanton': 'canton'})

    # Add year suffix
    for c in ['avg_premium','median_premium','std_premium','p10_premium','p90_premium','iqr_premium']:
        agg[f'{c}_{year_label}'] = agg.pop(c).round(1)
    agg = agg.rename(columns={'n_insurers': f'n_insurers_{year_label}'})

    print(f'✅ [{year_label}] {len(agg)} canton×region entries')
    return agg


p2025 = build_premium_features(PATH_PRIMES_2025, '2025')
p2026 = build_premium_features(PATH_PRIMES_2026, '2026')

# Merge both years
df_primes = p2025.merge(p2026, on=['canton', 'join_key'], how='outer')
df_primes['premium_increase_pct'] = (
    (df_primes['avg_premium_2026'] - df_primes['avg_premium_2025'])
    / df_primes['avg_premium_2025'] * 100
).round(2)
df_primes['n_insurers_change'] = df_primes['n_insurers_2026'] - df_primes['n_insurers_2025']

print(f'\nPremium table: {df_primes.shape[0]} rows × {df_primes.shape[1]} columns')
display(df_primes[['canton','join_key','avg_premium_2025','avg_premium_2026',
                    'std_premium_2026','iqr_premium_2026','n_insurers_2026',
                    'premium_increase_pct']].head(10))

## 4. Merge All Sources → Final Dataset

**Join sequence:**
1. OFS stats ← join on `bfs_nr` → OFSP labels (gets canton, premium_region, join_key)
2. Fill `join_key` for single-region cantons not in ordonnance (`canton_H0`)
3. ← join on `join_key` → Premium market features

In [ ]:
# Step 1: OFS + OFSP labels
df = pd.merge(
    df_ofs,
    df_labels[['bfs_nr','canton','commune','premium_region','join_key']],
    on='bfs_nr', how='left'
)

# Step 2: Fill canton for unmatched rows (single-region cantons)
df['canton'] = df.apply(
    lambda r: r['canton'] if pd.notna(r['canton'])
    else bfs_canton_lookup.get(r['bfs_nr']), axis=1
)

# Assign premium_region=1 and join_key for CH0 cantons
mask_ch0 = df['premium_region'].isna() & df['canton'].isin(CH0_CANTONS)
df.loc[mask_ch0, 'premium_region'] = 1
df.loc[mask_ch0, 'join_key'] = df.loc[mask_ch0, 'canton'] + '_H0'

print(f'After label join:')
print(f'  Communes with region:    {df["premium_region"].notna().sum()}')
print(f'  Communes without region: {df["premium_region"].isna().sum()}')
print(f'  Region distribution: {df["premium_region"].value_counts().sort_index().to_dict()}')

# Step 3: Join premium features via join_key
df = pd.merge(df, df_primes.drop(columns='canton'), on='join_key', how='left')

# Final cleanup
df_clean = df.dropna(subset=['premium_region']).copy()
df_clean['premium_region'] = df_clean['premium_region'].astype(int)

print(f'\n✅ Final dataset: {df_clean.shape[0]} communes × {df_clean.shape[1]} columns')
print(f'   premium NaN: {df_clean["avg_premium_2026"].isna().sum()}')

# Sanity check: GE vs BE must have different premiums
print('\n=== Sanity check: GE (CH0) vs BE (CH1/2/3) ===')
check = df_clean[df_clean['canton'].isin(['GE','BE'])].groupby(
    ['canton','premium_region','join_key'])['avg_premium_2026'].mean().round(1)
print(check)

## 5. Feature Engineering

In [ ]:
# Derived OFS features
df_clean['pct_large_hh']    = df_clean['pct_hh_4p'] + df_clean['pct_hh_5p_plus']
df_clean['pct_small_hh']    = df_clean['pct_hh_1p'] + df_clean['pct_hh_2p']
df_clean['log_pop_density'] = np.log1p(df_clean['pop_density_2024'])
df_clean['is_urban']        = (df_clean['pop_density_2024'] > 1000).astype(int)

# Feature groups
FEATURES_OFS = [
    'log_pop_density', 'pop_density_2024', 'is_urban',
    'pct_hh_1p', 'pct_hh_2p', 'pct_hh_3p', 'pct_hh_4p', 'pct_hh_5p_plus',
    'pct_large_hh', 'pct_small_hh',
    'social_assistance_rate', 'social_assistance_count',
]
FEATURES_PREMIUM = [
    'avg_premium_2025', 'avg_premium_2026',
    'median_premium_2026', 'std_premium_2026',
    'p10_premium_2026', 'p90_premium_2026',
    'iqr_premium_2026',
    'n_insurers_2026', 'premium_increase_pct', 'n_insurers_change',
]

FEATURE_COLS = [c for c in FEATURES_OFS + FEATURES_PREMIUM if c in df_clean.columns]

print(f'Total features: {len(FEATURE_COLS)}')
print(f'  OFS socio-demo: {len([c for c in FEATURE_COLS if c in FEATURES_OFS])}')
print(f'  Premium market: {len([c for c in FEATURE_COLS if c in FEATURES_PREMIUM])}')
print(f'\nTarget distribution:')
print(df_clean['premium_region'].value_counts().sort_index())

In [ ]:
# Impute missing values with median (robust)
imputer = SimpleImputer(strategy='median')
df_clean[FEATURE_COLS] = imputer.fit_transform(df_clean[FEATURE_COLS])

remaining_nan = df_clean[FEATURE_COLS].isnull().sum().sum()
print(f'✅ Imputation done — remaining NaN in features: {remaining_nan}')

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# 6.1 — Financial stakes: premium per region + increase per canton
palette = {1: '#e74c3c', 2: '#f39c12', 3: '#27ae60'}
region_labels = {1: 'Region 1\n(high cost)', 2: 'Region 2\n(medium)', 3: 'Region 3\n(low cost)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Financial Stakes of Premium Region Classification', fontsize=14, fontweight='bold')

# Left: mean premium per region (only multi-region cantons for fairness)
multi_region = df_clean[~df_clean['join_key'].str.endswith('H0')]
region_premium = multi_region.groupby('premium_region')['avg_premium_2026'].mean()
bars = axes[0].bar(region_premium.index,
                   region_premium.values,
                   color=[palette[r] for r in region_premium.index],
                   edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, region_premium.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+5,
                 f'CHF {val:.0f}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Premium Region')
axes[0].set_ylabel('Avg Monthly Premium (CHF)\nAdult, 300 CHF franchise, standard model')
axes[0].set_title('Mean Reference Premium per Region (2026)\n(multi-region cantons only)')
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels([region_labels[r] for r in [1, 2, 3]])

# Right: increase per canton (one bar per canton, averaged across regions)
canton_increase = df_clean.groupby('canton')['premium_increase_pct'].mean().sort_values()
mean_inc = canton_increase.mean()
axes[1].barh(canton_increase.index, canton_increase.values, color='#3498db', alpha=0.8)
axes[1].axvline(mean_inc, color='red', linestyle='--', linewidth=1.5,
                label=f'Mean: {mean_inc:.1f}%')
axes[1].set_xlabel('Avg Premium Increase 2025 → 2026 (%)')
axes[1].set_title('Premium Increase by Canton\n(standard adult profile)')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_financial_stakes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to outputs/')

In [ ]:
# 6.2 — Intra-canton gap: Berne example (strongest argument for the paper)
be = df_primes[df_primes['canton'] == 'BE'][['join_key','avg_premium_2026']].sort_values('join_key')
be_labels = {'BE_H1': 'Region 1 (urban)', 'BE_H2': 'Region 2 (peri-urban)', 'BE_H3': 'Region 3 (rural)'}

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#e74c3c', '#f39c12', '#27ae60']
bars = ax.bar([be_labels[k] for k in be['join_key']],
              be['avg_premium_2026'], color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, be['avg_premium_2026']):
    ax.text(bar.get_x()+bar.get_width()/2, val+3,
            f'CHF {val:.0f}', ha='center', fontweight='bold')

gap = be['avg_premium_2026'].max() - be['avg_premium_2026'].min()
ax.set_title(f'Canton of Berne: CHF {gap:.0f}/month gap between regions\n= CHF {gap*12:.0f}/year for the same coverage',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Avg Monthly Premium (CHF)')
ax.set_ylim(0, be['avg_premium_2026'].max() * 1.15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_berne_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 — Socio-demographic profiles by region
eda_cols   = ['pop_density_2024', 'pct_hh_1p', 'pct_large_hh', 'social_assistance_rate']
eda_titles = ['Population density\n(hab/km²)', '% Single-person\nhouseholds',
              '% Large households\n(4+ persons)', 'Social assistance\nrate (%)']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Socio-Demographic Profiles by Premium Region', fontsize=13, fontweight='bold')

for ax, col, title in zip(axes, eda_cols, eda_titles):
    for region in [1, 2, 3]:
        data = df_clean[df_clean['premium_region'] == region][col].dropna()
        ax.hist(data, bins=25, alpha=0.6, color=palette[region],
                label=region_labels[region], density=True)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_sociodemographic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6.4 — Correlation heatmap
corr_cols = FEATURE_COLS + ['premium_region']
corr = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.4, ax=ax,
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop correlations with premium_region:')
print(corr['premium_region'].drop('premium_region')
      .sort_values(key=abs, ascending=False).round(3).head(10))

In [ ]:
# 6.5 — Summary statistics per region
summary_cols = ['pop_density_2024','pct_hh_1p','pct_large_hh',
                'social_assistance_rate','avg_premium_2026','premium_increase_pct']
print('=== Mean feature values by premium region ===')
display(df_clean.groupby('premium_region')[summary_cols].mean().round(2))

## 7. Export Final ML-Ready Dataset

In [ ]:
ID_COLS    = ['bfs_nr', 'commune_ofs', 'canton', 'join_key']
TARGET_COL = ['premium_region']
AUDIT_COLS = [c for c in ['sa_estimated', 'sa_count_est'] if c in df_clean.columns]

final_cols = ID_COLS + TARGET_COL + FEATURE_COLS + AUDIT_COLS
df_final   = df_clean[final_cols].reset_index(drop=True)

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'✅ Exported: {OUTPUT_CSV}')
print(f'   Shape: {df_final.shape[0]} communes × {df_final.shape[1]} columns')
print(f'\nColumn overview:')
for i, col in enumerate(final_cols, 1):
    nan = df_final[col].isnull().sum()
    print(f'  {i:2d}. {col:<35} NaN={nan}')

display(df_final.head(5))

## 8. Ready for ML

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler

X = df_final[FEATURE_COLS]
y = df_final['premium_region']

# Stratified split — important because classes are imbalanced (332/824/347)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
```

**Next steps:**
1. **Unsupervised** — PCA (visualize communes) + KMeans k=3 (compare to official regions)
2. **Supervised** — LogisticRegression → RandomForest → XGBoost (tuned with GridSearchCV + 5-fold CV)
3. **Metrics** — Accuracy + **Cohen's Kappa** (required for multiclass imbalanced)
4. **Interpretability** — SHAP values + table of misclassified communes with CHF financial impact

## 9. Push to GitHub

From your terminal in the project root:
```bash
git add outputs/swiss_communes_ml_ready.csv
git add notebooks/swiss_health_insurance_ML_v3.ipynb
git commit -m "feat: add ML-ready dataset with corrected join key"
git push
```